In [1]:
#Importación de librerías
import pyodbc
import pandas as pd
import numpy as np

In [2]:
#Conexión a SQL Server
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=DESKTOP-87MLI8D\MSSQLSERVER01;"
    "DATABASE=Delitos Estatales;"
    "Trusted_Connection=yes;"
)

In [3]:
#Carga tabla original
query = "SELECT * FROM delitos_fuero_comun"
df = pd.read_sql(query, conn)

C:\Users\Nefi\AppData\Local\Temp\ipykernel_16152\692054870.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
#EDA
#TOP 20
df.head(20)

#Conteo de registros
total_registros = len(df)

#Rango de años
año_min = df['Año'].min()
año_max = df['Año'].max()

print(total_registros, año_min, año_max)

34496 2015 2025


In [5]:
#UNPIVOT equivalente en SQL con MELT
#Columnas de meses
meses = [
    'Enero','Febrero','Marzo','Abril','Mayo','Junio',
    'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre'
]

#Unpivot con melt
df_norm_py = df.melt(
    id_vars=[
        'Año','Clave_Ent','Entidad',
        'Bien_jurídico_afectado',
        'Tipo_de_delito',
        'Subtipo_de_delito',
        'Modalidad'
    ],
    value_vars=meses,
    var_name='Mes',
    value_name='Delitos'
)

df_norm_py.head()

,Año,Clave_Ent,Entidad,Bien_jurídico_afectado,Tipo_de_delito,Subtipo_de_delito,Modalidad,Mes,Delitos
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,Enero,3
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,Enero,1
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,Enero,0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,Enero,2
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,Enero,0


In [6]:
#Variables derivadas de meses
mes_map = {
    'Enero':1,'Febrero':2,'Marzo':3,'Abril':4,'Mayo':5,'Junio':6,
    'Julio':7,'Agosto':8,'Septiembre':9,'Octubre':10,'Noviembre':11,'Diciembre':12
}

df_norm_py['Mes_num'] = df_norm_py['Mes'].map(mes_map)

In [7]:
#Asiganción de Fecha
df_norm_py['Fecha'] = pd.to_datetime(
    df_norm_py['Año'].astype(str) + '-' +
    df_norm_py['Mes_num'].astype(str) + '-01'
)

In [8]:
#Formato Año-Mes
df_norm_py['AñoMes'] = df_norm_py['Fecha'].dt.strftime('%Y-%m')

In [9]:
#Definición de temporadas estacionales
def temporada(mes):
    if mes in [12,1,2]:
        return 'Invierno'
    elif mes in [3,4,5]:
        return 'Primavera'
    elif mes in [6,7,8]:
        return 'Verano'
    else:
        return 'Otoño'

df_norm_py['Temporada'] = df_norm_py['Mes_num'].apply(temporada)

In [10]:
df_norm_py.head()

,Año,Clave_Ent,Entidad,Bien_jurídico_afectado,Tipo_de_delito,Subtipo_de_delito,Modalidad,Mes,Delitos,Mes_num,Fecha,AñoMes,Temporada
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,Enero,3,1,2015-01-01,2015-01,Invierno
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,Enero,1,1,2015-01-01,2015-01,Invierno
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,Enero,0,1,2015-01-01,2015-01,Invierno
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,Enero,2,1,2015-01-01,2015-01,Invierno
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,Enero,0,1,2015-01-01,2015-01,Invierno


In [11]:
#Promedio de incidencia por estado y mes
df_promedio_estado_mes_py = (
    df_norm_py
    .groupby(['Entidad','AñoMes'])['Delitos']
    .mean()
    .reset_index(name='Promedio_Delitos')
)

In [12]:
df_promedio_estado_mes_py.head()

,Entidad,AñoMes,Promedio_Delitos
0,Aguascalientes,2015-01,18.928571
1,Aguascalientes,2015-02,18.530612
2,Aguascalientes,2015-03,20.500000
3,Aguascalientes,2015-04,20.183673
4,Aguascalientes,2015-05,19.755102


In [13]:
#Delito más común por estado
df_delito_estado_py = (
    df_norm_py
    .groupby(['Entidad','Tipo_de_delito'])['Delitos']
    .sum()
    .reset_index(name='Total_Delitos')
)

#Ranking
df_delito_estado_py['ranking'] = (
    df_delito_estado_py
    .sort_values(['Entidad','Total_Delitos'], ascending=[True, False])
    .groupby('Entidad')
    .cumcount() + 1
)

#Top 1 por estado
df_top_delito_py = df_delito_estado_py[df_delito_estado_py['ranking'] == 1]

In [15]:
df_delito_estado_py.head()

,Entidad,Tipo_de_delito,Total_Delitos,ranking
0,Aguascalientes,Aborto,91,30
1,Aguascalientes,Abuso de confianza,7575,9
2,Aguascalientes,Abuso sexual,19,35
3,Aguascalientes,Acoso sexual,0,39
4,Aguascalientes,Allanamiento de morada,5194,11


In [16]:
df_top_delito_py.head()

,Entidad,Tipo_de_delito,Total_Delitos,ranking
32,Aguascalientes,Robo,127917,1
72,Baja California,Robo,402897,1
112,Baja California Sur,Robo,83021,1
152,Campeche,Robo,23048,1
192,Chiapas,Robo,57004,1


In [ ]:
#Delitos por temporada
df_temporada_py = (
    df_norm_py
    .groupby('Temporada')['Delitos']
    .sum()
    .reset_index(name='Total_Delitos')
    .sort_values('Total_Delitos', ascending=False)
)

In [18]:
df_temporada_py.head()

,Temporada,Total_Delitos
2,Primavera,5557346
3,Verano,5545388
1,Otoño,5514234
0,Invierno,5115462


In [19]:
#Promedio en Puebla por tipo y mes
df_puebla_py = (
    df_norm_py[df_norm_py['Entidad'] == 'Puebla']
    .groupby(['Año','Mes','Tipo_de_delito'])['Delitos']
    .mean()
    .reset_index(name='promedio_delitos')
)

In [20]:
df_puebla_py.head()

,Año,Mes,Tipo_de_delito,promedio_delitos
0,2015,Abril,Aborto,0.0
1,2015,Abril,Abuso de confianza,95.0
2,2015,Abril,Abuso sexual,0.0
3,2015,Abril,Acoso sexual,9.0
4,2015,Abril,Allanamiento de morada,39.0


In [21]:
#Indice de riesgo
df_riesgo_py = (
    df_norm_py
    .groupby('Entidad')
    .agg(
        total_delitos=('Delitos','sum'),
        promedio_mensual=('Delitos','mean'),
        meses_registrados=('AñoMes','nunique')
    )
    .reset_index()
    .sort_values('total_delitos', ascending=False)
)

In [22]:
#Datos para simulación de monte carlo
df_montecarlo_py = (
    df_norm_py
    .groupby(['Entidad','AñoMes'])['Delitos']
    .sum()
    .reset_index(name='total_delitos')
    .sort_values(['Entidad','AñoMes'])
)